# 01 - Data Profiling
Profiles every RAW synthetic dataset before any cleaning happens, so every cleaning decision in `02_data_cleaning.ipynb` is backed by an actual measurement.

**All data used here is synthetic** (see `data/raw/track3_dataset_notes.txt`).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd
from src import ingestion, profiling
pd.set_option('display.max_columns', 50)

## Load raw datasets

In [2]:
raw = ingestion.load_all_raw()
{name: df.shape for name, df in raw.items()}

{'mandi_master': (60, 6),
 'arrivals': (12180, 8),
 'prices': (9000, 9),
 'msp': (7, 4),
 'weather': (6000, 7),
 'transport': (8000, 10)}

## Raw row counts (for Gate 1 proof-of-cleaning comparison later)

In [3]:
raw_counts = {name: len(df) for name, df in raw.items()}
raw_counts

{'mandi_master': 60,
 'arrivals': 12180,
 'prices': 9000,
 'msp': 7,
 'weather': 6000,
 'transport': 8000}

## Missingness per dataset

In [4]:
for name, df in raw.items():
    print(f'--- {name} ---')
    display(profiling.profile_missingness(df).head(10))

--- mandi_master ---


,missing_count,missing_pct
mandi_type,9,15.00
district,6,10.00
total_area_acres,2,3.33
mandi_id,0,0.00
mandi_name,0,0.00
state,0,0.00


--- arrivals ---


,missing_count,missing_pct
variety,1696,13.92
farmer_count,981,8.05
unit,458,3.76
arrival_quantity,357,2.93
arrival_id,61,0.50
crop_name,0,0.00
date,0,0.00
mandi_id,0,0.00


--- prices ---


...

,missing_count,missing_pct
crop_name,0,0.0
season,0,0.0
year,0,0.0
msp_per_quintal,0,0.0


--- weather ---


,missing_count,missing_pct
rain_unit,636,10.60
temp_unit,584,9.73
humidity_percent,356,5.93
rainfall,287,4.78
temperature,232,3.87
sensor_id,123,2.05
timestamp,0,0.00


--- transport ---


,missing_count,missing_pct
vehicle_no,1216,15.20
distance_unit,1211,15.14
transit_hours,677,8.46
departure_time,425,5.31
arrival_time,393,4.91
driver_id,159,1.99
mandi_id,0,0.00
trip_id,0,0.00
destination_warehouse,0,0.00
distance,0,0.00


## Duplicate counts per dataset

In [5]:
for name, df in raw.items():
    print(name, profiling.profile_duplicates(df))

mandi_master {'exact_duplicates': 3, 'business_key_duplicates': None}
arrivals {'exact_duplicates': 180, 'business_key_duplicates': None}
prices {'exact_duplicates': 0, 'business_key_duplicates': None}
msp {'exact_duplicates': 0, 'business_key_duplicates': None}
weather {'exact_duplicates': 0, 'business_key_duplicates': None}
transport {'exact_duplicates': 0, 'business_key_duplicates': None}


## Observed messiness (spot checks)
A few manual spot-checks that motivate the standardization functions in `src/standardization.py`.

In [6]:
print('Distinct raw crop_name values (sample):')
print(sorted(raw['arrivals']['crop_name'].dropna().unique())[:20])

print('\nDistinct raw quantity units:')
print(raw['arrivals']['unit'].dropna().unique())

print('\nDistinct raw mandi_id formats (sample):')
print(raw['arrivals']['mandi_id'].dropna().unique()[:15])

Distinct raw crop_name values (sample):
['Basmati', 'Chawal', 'Corn', 'Cotton', 'GEHUN', 'Ganna', 'Ganne', 'Gehun', 'Kapas', 'Maize', 'Makka', 'Mustard', 'Paddy', 'Rice', 'Sarso', 'Sarson', 'Sugarcane', 'WHEAT', 'Wheat', 'basmati']

Distinct raw quantity units:
<ArrowStringArray>
[      'MT',     'Kilo',       'kg',        'Q',   'Tonnes',      'QTL',
      'qtl',       'Kg',   'tonnes',        'T',       'KG',      'KGS',
 'Quintals',      'Qtl',  'Quintal']
Length: 15, dtype: str

Distinct raw mandi_id formats (sample):
<ArrowStringArray>
[      '036',       '043',       '016', 'MANDI-045',  'mandi055',      'M029',
      'M035',  'MANDI047', 'mandi_017', 'MANDI-023',      'M043',  'MANDI020',
  'MANDI055', 'MANDI-007',  'MANDI050']
Length: 15, dtype: str


These findings directly motivate: `normalize_crop()`, `normalize_mandi_id()`, `convert_to_quintal()`, `parse_date_any()` and `normalize_timezone()` in `src/standardization.py`, applied next in `02_data_cleaning.ipynb`.